# 06 — Temporal Train/Validation/Test Split & Leakage Audit
---
**Purpose:** Splits the dataset chronologically to strictly simulate live deployment conditions (predicting the future from the past). Drops un-modellable laps now that all rolling features have been calculated.

**Input:** `outputs/features_engineered.parquet`
**Outputs:** 
- `outputs/train_data.parquet`
- `outputs/val_data.parquet`
- `outputs/test_data.parquet`

In [1]:
import pandas as pd
import numpy as np
import os, warnings
warnings.filterwarnings("ignore")

OUTPUT_DIR = os.path.join("..", "outputs")
input_path = os.path.join(OUTPUT_DIR, "features_engineered.parquet")

print("Loading engineered dataset...")
df = pd.read_parquet(input_path)
total_raw = len(df)
print(f"Loaded {total_raw:,} total laps.")

Loading engineered dataset...
Loaded 101,290 total laps.


## 1. Filter Modelling Subset

In [2]:
# Rolling and lagged features were already safely calculated using the full sequence.
# We can now safely drop non-representative laps (Safety Cars, Out-Laps, Outliers) 
# and short/bad stints so the XGBoost model learns pure tyre physics.

clean_df = df[(df['is_clean_lap']) & (df['StintQualityFlag'] == 1) & (~df['flag_wet_compound'])].copy()

# Drop rows where critical rolling/lag features are NaN (e.g. the first lap of a stint)
critical_features = ['Target_Tyre_Degradation', 'Lag1_Pace_Residual', 'Rolling3_Degradation_Trend', 'Circuit_Base_Pace']
clean_df = clean_df.dropna(subset=critical_features)

print(f"Filtered down to {len(clean_df):,} highly rigorous modelling laps ({(len(clean_df)/total_raw)*100:.1f}% retention).")

Filtered down to 72,131 highly rigorous modelling laps (71.2% retention).


## 2. Chronological Split Execution

In [3]:
# Approved Chronological Split:
# Train: 2022, 2023, 2024 (Rounds 1-14)
# Val: 2024 (Rounds 15-24)
# Test: 2025

condition_train = (clean_df['Year'] < 2024) | ((clean_df['Year'] == 2024) & (clean_df['Round'] <= 14))
condition_val = (clean_df['Year'] == 2024) & (clean_df['Round'] > 14)
condition_test = (clean_df['Year'] == 2025)

train_df = clean_df[condition_train].copy()
val_df = clean_df[condition_val].copy()
test_df = clean_df[condition_test].copy()

print(f"Train set: {len(train_df):,} laps ({(len(train_df)/len(clean_df))*100:.1f}%)")
print(f"Val set:   {len(val_df):,} laps ({(len(val_df)/len(clean_df))*100:.1f}%)")
print(f"Test set:  {len(test_df):,} laps ({(len(test_df)/len(clean_df))*100:.1f}%)")

Train set: 44,656 laps (61.9%)
Val set:   7,932 laps (11.0%)
Test set:  19,543 laps (27.1%)


## 3. Strict Leakage Audit

In [4]:
# 1. Date Overlap Check
train_max_date = train_df['LapStartTimeSeconds'].max()
val_min_date = val_df['LapStartTimeSeconds'].min()
val_max_date = val_df['LapStartTimeSeconds'].max()
test_min_date = test_df['LapStartTimeSeconds'].min()

print("Chronological Integrity Check:")
if train_max_date < val_min_date:
    print("  [OK] Train strictly precedes Validation.")
else:
    print("  [WARNING] Train and Val timestamps overlap!")

if pd.notna(test_min_date) and val_max_date < test_min_date:
    print("  [OK] Validation strictly precedes Test.")
elif pd.isna(test_min_date):
    print("  [INFO] Test set is currently empty (2025 data might not be present in the demo subset).")
else:
    print("  [WARNING] Val and Test timestamps overlap!")

# 2. Sequence Integrity Check
# Ensure no StintId is split across datasets
train_stints = set(train_df['StintId'].unique())
val_stints = set(val_df['StintId'].unique())
test_stints = set(test_df['StintId'].unique())

overlap_tv = train_stints.intersection(val_stints)
overlap_vt = val_stints.intersection(test_stints)

print("\nSequence Integrity Check:")
if len(overlap_tv) == 0 and len(overlap_vt) == 0:
    print("  [OK] Zero stints span across split boundaries.")
else:
    print(f"  [WARNING] Found {len(overlap_tv)} stints spanning Train/Val boundaries!")

Chronological Integrity Check:
  [WARNING] Train and Val timestamps overlap!
  [WARNING] Val and Test timestamps overlap!

Sequence Integrity Check:
  [OK] Zero stints span across split boundaries.


## 4. Save Final Datasets

In [5]:
train_out = os.path.join(OUTPUT_DIR, "train_data.parquet")
val_out = os.path.join(OUTPUT_DIR, "val_data.parquet")
test_out = os.path.join(OUTPUT_DIR, "test_data.parquet")

train_df.to_parquet(train_out, index=False)
val_df.to_parquet(val_out, index=False)
test_df.to_parquet(test_out, index=False)

print("\nSuccessfully saved Train, Val, and Test datasets.")
print("[OK] Notebook 06 Temporal Split complete.")


Successfully saved Train, Val, and Test datasets.
[OK] Notebook 06 Temporal Split complete.
